# E0006 — reusable runtime image builder (NO GPU)

Purpose: materialize the already validated CUDA 12.9 vLLM runtime as a reusable Kaggle notebook output so future L4 runs do not spend GPU time reinstalling ~194 wheels.

Required settings:
- Accelerator: **None**
- Internet: **OFF**
- Attach exactly one saved wheelhouse output containing `e0006_cu129_wheelhouse/`

Expected outputs:
- `/kaggle/working/e0006_runtime/`
- `/kaggle/working/e0006_runtime_image_manifest.json`


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
RUNTIME = Path('/kaggle/working/e0006_runtime')
OUT = Path('/kaggle/working/e0006_runtime_image_manifest.json')
EXPECTED_VLLM_SHA256 = 'bf0d52faa2a51e7a01c6856a7a8a2d1307fd0ff711415d34168a67ffac0fa47b'
EXPECTED_CUBIN_SHA256 = 'c79fba990aee2a7c7ef64208bb65900e45fe23c3a223f3dfc21eef225f43cba2'

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def tree_bytes(path: Path) -> int:
    total = 0
    for p in path.rglob('*'):
        if p.is_file():
            try: total += p.stat().st_size
            except OSError: pass
    return total

vllm_hits = sorted(INPUT_ROOT.rglob('vllm-0.27.1+cu129-*.whl'))
cubin_hits = sorted(INPUT_ROOT.rglob('flashinfer_cubin-0.6.16.post3-*.whl'))
payload = {'experiment':'E0006','gate':'D5_REUSABLE_RUNTIME_IMAGE_NO_GPU','python':sys.version,'status':'STARTING'}

if len(vllm_hits) != 1 or len(cubin_hits) != 1:
    payload['status'] = 'BLOCKED_INPUT'
    payload['vllm_hits'] = [str(x) for x in vllm_hits]
    payload['cubin_hits'] = [str(x) for x in cubin_hits]
    OUT.write_text(json.dumps(payload, indent=2, sort_keys=True)+'\n')
    raise SystemExit('Attach exactly one wheelhouse output.')

vllm_wheel = vllm_hits[0]
cubin_wheel = cubin_hits[0]
wheelhouse = vllm_wheel.parent
payload['wheelhouse'] = str(wheelhouse)
payload['wheelhouse_file_count'] = len(list(wheelhouse.glob('*.whl')))
payload['vllm_sha256'] = sha256(vllm_wheel)
payload['flashinfer_cubin_sha256'] = sha256(cubin_wheel)
payload['hashes_ok'] = payload['vllm_sha256'] == EXPECTED_VLLM_SHA256 and payload['flashinfer_cubin_sha256'] == EXPECTED_CUBIN_SHA256
payload['free_working_gib_before'] = round(shutil.disk_usage('/kaggle/working').free / 1024**3, 3)
if payload['free_working_gib_before'] < 16.0:
    payload['status'] = 'BLOCKED_DISK'
    OUT.write_text(json.dumps(payload, indent=2, sort_keys=True)+'\n')
    raise SystemExit('Need >=16 GiB free in /kaggle/working.')

if RUNTIME.exists(): shutil.rmtree(RUNTIME)
RUNTIME.mkdir(parents=True)
cmd = [sys.executable,'-m','pip','install','--no-index','--find-links',str(wheelhouse),'--target',str(RUNTIME),'--ignore-installed','--no-cache-dir','--no-compile',str(vllm_wheel),'flashinfer-cubin==0.6.16.post3']
payload['install_command'] = cmd
t0=time.time()
cp=subprocess.run(cmd,capture_output=True,text=True,timeout=2400)
payload['install_seconds']=round(time.time()-t0,3)
payload['install_returncode']=cp.returncode
payload['install_stderr_tail']=cp.stderr[-8000:]
if cp.returncode != 0:
    payload['status']='BLOCKED_INSTALL'
    OUT.write_text(json.dumps(payload,indent=2,sort_keys=True)+'\n')
    raise SystemExit('Offline runtime materialization failed.')

probe_code = r'''
import importlib.metadata as md, json
import torch, transformers, flashinfer, vllm, triton
from vllm.model_executor.models.nemotron_h import NemotronHForCausalLM
print(json.dumps({'torch':md.version('torch'),'vllm':md.version('vllm'),'transformers':md.version('transformers'),'triton':md.version('triton'),'flashinfer-python':md.version('flashinfer-python'),'flashinfer-cubin':md.version('flashinfer-cubin'),'torch_cuda_build':torch.version.cuda,'nemotron_class':NemotronHForCausalLM.__name__},sort_keys=True))
'''
env=os.environ.copy()
env['PYTHONPATH']=str(RUNTIME)
env['PATH']=str(RUNTIME/'bin')+os.pathsep+env.get('PATH','')
env['HF_HUB_OFFLINE']='1'; env['TRANSFORMERS_OFFLINE']='1'; env['VLLM_NO_USAGE_STATS']='1'
probe=subprocess.run([sys.executable,'-c',probe_code],capture_output=True,text=True,env=env,timeout=300)
payload['probe_returncode']=probe.returncode
payload['probe_stdout']=probe.stdout[-8000:]
payload['probe_stderr']=probe.stderr[-8000:]
probe_payload=None
if probe.stdout.strip():
    try: probe_payload=json.loads(probe.stdout.strip().splitlines()[-1])
    except Exception: pass
payload['probe']=probe_payload
payload['runtime_installed_bytes']=tree_bytes(RUNTIME)
payload['runtime_installed_gib']=round(payload['runtime_installed_bytes']/1024**3,3)
payload['free_working_gib_after']=round(shutil.disk_usage('/kaggle/working').free/1024**3,3)
versions_ok = bool(probe_payload) and probe_payload.get('torch')=='2.13.0+cu129' and probe_payload.get('vllm')=='0.27.1+cu129' and probe_payload.get('flashinfer-python')=='0.6.16.post3' and probe_payload.get('flashinfer-cubin')=='0.6.16.post3' and probe_payload.get('torch_cuda_build')=='12.9' and probe_payload.get('nemotron_class')=='NemotronHForCausalLM'
payload['status']='RUNTIME_IMAGE_READY' if payload['hashes_ok'] and cp.returncode==0 and probe.returncode==0 and versions_ok else 'BLOCKED_RUNTIME_IMAGE'
payload['next_rule']='Attach e0006_runtime as read-only input to Gate B; do not reinstall wheels during L4 runs.'
OUT.write_text(json.dumps(payload,indent=2,sort_keys=True)+'\n')
print(json.dumps({'status':payload['status'],'runtime_installed_gib':payload['runtime_installed_gib'],'free_working_gib_after':payload['free_working_gib_after'],'probe':payload['probe']},indent=2,sort_keys=True))
print(f'\nWROTE: {OUT}')
